# Phase 3 LoRA Anomaly Explainer

目标：使用 `Qwen/Qwen2.5-0.5B-Instruct` 对确定性异常事实做 LoRA SFT。模型只生成说明，不计算金额、不决定风险等级、不改变建议动作。默认只运行 bootstrap 与 dry-run smoke，不启动训练。

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

from procureguard.phase3.paths import resolve_project_root
from procureguard.phase3.gpu_notebook import (
    notebook_kernel_python_from_env,
    notebook_model_dir_from_env,
)

PROJECT_ROOT = resolve_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Phase 3D configuration switches. Only change these booleans intentionally.
RUN_TRAINING = False
RUN_FULL_EVAL = False

# Base smoke can be enabled after guard passes; full comparison is controlled by RUN_FULL_EVAL.
RUN_BASE_SMOKE = False
REQUIRE_CUDA_FOR_TRAINING = True
PHASE3_MODEL_DIR = notebook_model_dir_from_env()
EXPECTED_KERNEL_PYTHON = notebook_kernel_python_from_env()
if EXPECTED_KERNEL_PYTHON and Path(sys.executable).resolve() != Path(EXPECTED_KERNEL_PYTHON).resolve():
    raise RuntimeError(f'Notebook Kernel Python mismatch: {sys.executable} != {EXPECTED_KERNEL_PYTHON}')
print({
    'project_root': str(PROJECT_ROOT),
    'python': sys.executable,
    'expected_kernel_python': EXPECTED_KERNEL_PYTHON,
    'phase3_model_dir': PHASE3_MODEL_DIR,
    'run_training': RUN_TRAINING,
    'run_base_smoke': RUN_BASE_SMOKE,
    'run_full_eval': RUN_FULL_EVAL,
})


## Bootstrap Guard

可写 bootstrap 会创建 `artifacts/phase3/` 子目录，校验数据 SHA、依赖、CUDA、模型目录，并导出 `environment_guard.json`。

In [ ]:
from procureguard.phase3.gpu_notebook import notebook_runtime_guard, phase3_paths

paths = phase3_paths(PROJECT_ROOT, model_dir=PHASE3_MODEL_DIR)
guard = notebook_runtime_guard(
    PROJECT_ROOT,
    require_cuda=RUN_TRAINING and REQUIRE_CUDA_FOR_TRAINING,
    model_dir=PHASE3_MODEL_DIR,
    expected_kernel_python=EXPECTED_KERNEL_PYTHON,
)
print(json.dumps(guard, ensure_ascii=False, indent=2))
if RUN_TRAINING and not guard['training_ready']:
    raise RuntimeError(f"RUN_TRAINING=True but notebook runtime guard failed: {guard['failed_checks']}")


## Runtime Context

当前 Kernel 通过统一 runtime context 恢复数据、prompt、训练参数、LoRA 参数、生成参数和输出目录。

In [ ]:
from procureguard.phase3.gpu_notebook import hydrate_runtime_context
from procureguard.phase3.runtime import (
    format_sft_row,
    runtime_config_dict,
    to_messages,
    write_artifacts_manifest,
    write_predictions_jsonl,
)

runtime = hydrate_runtime_context(
    PROJECT_ROOT,
    model_dir=PHASE3_MODEL_DIR,
    require_cuda=RUN_TRAINING and REQUIRE_CUDA_FOR_TRAINING,
    expected_kernel_python=EXPECTED_KERNEL_PYTHON,
)
context = runtime['context']
config = runtime['config']
print(json.dumps(config, ensure_ascii=False, indent=2))

## Backend Selection

优先使用 Unsloth；不可用时使用 Transformers + PEFT + TRL。两条路径共用同一数据、prompt、seed、LoRA 参数和导出目录。

In [ ]:
from procureguard.phase3.gpu_notebook import run_base_inference_smoke

base_smoke = run_base_inference_smoke(
    PROJECT_ROOT,
    model_dir=PHASE3_MODEL_DIR,
    sample_count=1,
    run=RUN_BASE_SMOKE,
)
print(json.dumps(base_smoke, ensure_ascii=False, indent=2))

In [ ]:
model = tokenizer = None
if RUN_TRAINING and context.backend == 'unsloth':
    from unsloth import FastLanguageModel
    lora_kwargs = dict(config['lora'])
    lora_kwargs['target_modules'] = list(lora_kwargs['target_modules'])
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=PHASE3_MODEL_DIR or context.model_id,
        max_seq_length=context.training_config.max_seq_length,
        load_in_4bit=True,
    )
    model = FastLanguageModel.get_peft_model(
        model, use_gradient_checkpointing='unsloth', **lora_kwargs
    )
elif RUN_TRAINING:
    import torch
    from peft import LoraConfig, get_peft_model
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    quantization_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)
    model_source = PHASE3_MODEL_DIR or context.model_id
    tokenizer = AutoTokenizer.from_pretrained(model_source, use_fast=True, local_files_only=bool(PHASE3_MODEL_DIR))
    model = AutoModelForCausalLM.from_pretrained(model_source, device_map='auto', quantization_config=quantization_config, local_files_only=bool(PHASE3_MODEL_DIR))
    lora_kwargs = dict(config['lora'])
    lora_kwargs['target_modules'] = list(lora_kwargs['target_modules'])
    model = get_peft_model(model, LoraConfig(task_type='CAUSAL_LM', **lora_kwargs))
print('model load skipped' if not RUN_TRAINING else f'model ready via {context.backend}')

## SFT Training And Adapter Export

真实训练只在 `RUN_TRAINING=True` 且 guard 通过时执行；adapter、trainer log 和 checkpoint 都写入 `artifacts/phase3/`。

In [ ]:
trainer = None
if RUN_TRAINING:
    from datasets import Dataset
    from transformers import TrainingArguments
    from trl import SFTTrainer
    train_dataset = Dataset.from_list([format_sft_row(tokenizer, row) for row in context.train_rows])
    validation_dataset = Dataset.from_list([format_sft_row(tokenizer, row) for row in context.validation_rows])
    training_args = dict(config['training'])
    training_args.pop('max_seq_length')
    arguments = TrainingArguments(
        output_dir=str(paths.trainer_dir),
        report_to='none',
        per_device_eval_batch_size=2,
        bf16=True,
        **training_args,
    )
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_dataset,
        eval_dataset=validation_dataset,
        dataset_text_field='text',
        max_seq_length=context.training_config.max_seq_length,
        args=arguments,
    )
    trainer.train()
    trainer.save_model(str(paths.adapter_dir))
    tokenizer.save_pretrained(str(paths.adapter_dir))
    (paths.log_dir / 'trainer_log_history.json').write_text(json.dumps(trainer.state.log_history, indent=2), encoding='utf-8')
print('training skipped' if not RUN_TRAINING else f'adapter saved to {paths.adapter_dir}')

## Base Inference Smoke

默认只输出可执行计划；只有 `RUN_BASE_SMOKE=True` 且本地 `PHASE3_MODEL_DIR` 可用时，才对 test split 的少量样本执行 base 推理。

## Unified Evaluation

只有真实 `base.jsonl` 和 `fine_tuned.jsonl` 同时存在时才生成对比报告；本 Notebook 不写占位指标。

In [ ]:
import gc

base_predictions = paths.prediction_dir / 'base.jsonl'
fine_tuned_predictions = paths.prediction_dir / 'fine_tuned.jsonl'
evaluation_json = paths.evaluation_dir / 'evaluation.json'
evaluation_md = paths.evaluation_dir / 'evaluation.md'
manifest_path = paths.artifact_dir / 'artifacts_manifest.json'

def generate_prediction_file(adapter_path, output_path):
    import torch
    from peft import PeftModel
    from transformers import AutoModelForCausalLM, AutoTokenizer

    model_source = PHASE3_MODEL_DIR or context.model_id
    local_only = bool(PHASE3_MODEL_DIR)
    tokenizer = AutoTokenizer.from_pretrained(model_source, use_fast=True, local_files_only=local_only)
    inference_model = AutoModelForCausalLM.from_pretrained(
        model_source,
        device_map='auto' if torch.cuda.is_available() else None,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        local_files_only=local_only,
    )
    if adapter_path is not None:
        inference_model = PeftModel.from_pretrained(inference_model, str(adapter_path))
    inference_model.eval()

    explanations = []
    for row in context.test_rows:
        prompt = tokenizer.apply_chat_template(
            to_messages(row, include_answer=False),
            tokenize=False,
            add_generation_prompt=True,
        )
        inputs = tokenizer(prompt, return_tensors='pt').to(inference_model.device)
        with torch.no_grad():
            output_ids = inference_model.generate(
                **inputs,
                max_new_tokens=context.generation_config.max_new_tokens,
                do_sample=context.generation_config.do_sample,
            )
        new_tokens = output_ids[0, inputs['input_ids'].shape[-1]:]
        explanations.append(tokenizer.decode(new_tokens, skip_special_tokens=True).strip())
    write_predictions_jsonl(output_path, context.test_rows, explanations)
    del inference_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

if RUN_FULL_EVAL:
    if not guard['training_ready']:
        raise RuntimeError('RUN_FULL_EVAL=True but notebook guard is not ready')
    if not paths.adapter_dir.exists():
        raise RuntimeError('RUN_FULL_EVAL=True but adapter directory is missing')
    generate_prediction_file(None, base_predictions)
    generate_prediction_file(paths.adapter_dir, fine_tuned_predictions)

if base_predictions.exists() and fine_tuned_predictions.exists():
    command = [
        sys.executable,
        str(PROJECT_ROOT / 'scripts' / 'phase3' / 'evaluate_explanations.py'),
        '--dataset', str(paths.data_dir / 'test.jsonl'),
        '--base-predictions', str(base_predictions),
        '--fine-tuned-predictions', str(fine_tuned_predictions),
        '--output-dir', str(paths.evaluation_dir),
    ]
    subprocess.run(command, check=True)
else:
    print('No real base/fine-tuned prediction files yet; evaluation metrics are not generated.')

manifest = write_artifacts_manifest(
    manifest_path,
    {
        'environment_guard': paths.log_dir / 'environment_guard.json',
        'notebook_runtime_guard': paths.log_dir / 'notebook_runtime_guard.json',
        'training_config': paths.log_dir / 'training_config.json',
        'trainer_log_history': paths.log_dir / 'trainer_log_history.json',
        'base_predictions': base_predictions,
        'fine_tuned_predictions': fine_tuned_predictions,
        'evaluation_json': evaluation_json,
        'evaluation_md': evaluation_md,
    },
    adapter_dir=paths.adapter_dir,
)
print(json.dumps(manifest, ensure_ascii=False, indent=2))
